# Ejercicio 1

Construya una red de Kohonen de 2 entradas que aprenda una distribución uniforme dentro del círculo unitario. Mostrar el mapa de preservación de topología. Probar con distribuciones uniformes dentro de otras figuras geométricas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(110007)
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Computer Modern"
})

In [ ]:
def visualize_dataset(dataset):
    plt.figure(figsize=(3, 3))
    plt.scatter(dataset[:, 0], dataset[:, 1], alpha=0.2, label='Dataset')
    plt.xlim((-1.1, 1.1))
    plt.ylim((-1.1, 1.1))
    plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
n_samples = 1000

theta = np.random.uniform(low=0, high=2*np.pi, size=n_samples)
r = np.sqrt(np.random.uniform(low=0, high=1, size=n_samples))
data_circle = np.c_[r * np.cos(theta), r * np.sin(theta)]

visualize_dataset(data_circle)

In [ ]:
class KohonenNet:
    def __init__(self, n_neurons=(10, 10)):
        self.weights = np.random.uniform(low=-1.0, high=1.0, size=(n_neurons[0], n_neurons[1], 2))
        self.n_neurons = n_neurons
        self.grid_x, self.grid_y = np.meshgrid(np.arange(n_neurons[0]), np.arange(n_neurons[1]), indexing='ij')
        self.saved_weights = []

    def train(self, X, epochs=100, lr=0.01, sigma_sq_start=1.0, save_epochs=None):
        n_samples = X.shape[0]
        
        for epoch in range(epochs):
            idx = np.random.permutation(n_samples)
            sigma_sq = sigma_sq_start * (1 - epoch / epochs)

            if epoch + 1 in save_epochs:
                self.saved_weights.append(np.copy(self.weights))

            for i in idx:
                x = X[i]
                dist_sq = np.sum((self.weights - x) ** 2, axis=2)
                i_star = np.unravel_index(np.argmin(dist_sq), dist_sq.shape)
                dist_sq_grid = (self.grid_x - i_star[0])**2 + (self.grid_y - i_star[1])**2
                neighbourhood = np.exp(-dist_sq_grid / (2 * sigma_sq))
                delta_weights = lr * neighbourhood[:, :, np.newaxis] * (x - self.weights)
                self.weights += delta_weights

In [ ]:
def visualize(epochs, weight_list, dataset, name):
    n_subplots = len(epochs)
    plt.figure(figsize=(3.1 * n_subplots, 3))

    for k in range(n_subplots):
        ax = plt.subplot(1, n_subplots, k + 1)
        
        ax.scatter(dataset[:, 0], dataset[:, 1], alpha=0.2, label='Dataset')

        n_neurons = weight_list[k].shape
        weights = weight_list[k]
        
        for i in range(n_neurons[0]):
            for j in range(n_neurons[1] - 1):
                ax.plot([weights[i, j, 0], weights[i, j+1, 0]], 
                         [weights[i, j, 1], weights[i, j+1, 1]], 'r-', alpha=0.5)
        
        for i in range(n_neurons[0] - 1):
            for j in range(n_neurons[1]):
                ax.plot([weights[i, j, 0], weights[i+1, j, 0]], 
                         [weights[i, j, 1], weights[i+1, j, 1]], 'r-', alpha=0.5)
        
        weights_flat = weight_list[k].reshape(-1, 2)
        ax.scatter(weights_flat[:, 0], weights_flat[:, 1], c='tab:red', s=100, edgecolors='black', label='Neuronas')

        ax.axis('square')
        ax.set_xlim((-1.1, 1.1))
        ax.set_ylim((-1.1, 1.1))
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')
        ax.set_title(f'Época {epochs[k]}')
        ax.legend(loc='upper right')
        ax.grid()

    plt.tight_layout()
    plt.savefig(f'../informe/img/ej1/{name}.svg')
    plt.show()

In [ ]:
save_epochs = [1, 2, 300, 800]
model = KohonenNet(n_neurons=(5, 5))
model.train(data_circle, sigma_sq_start=1.0, lr=0.1, epochs=800, save_epochs=save_epochs)
visualize(save_epochs, model.saved_weights, data_circle, 'circle')

In [ ]:
data_square = np.random.uniform(low=-1, high=1, size=(n_samples, 2))
visualize_dataset(data_square)

In [ ]:
save_epochs = [1, 2, 300, 1000]
model = KohonenNet(n_neurons=(5, 5))
model.train(data_square, sigma_sq_start=1.0, lr=0.01, epochs=1000, save_epochs=save_epochs)
visualize(save_epochs, model.saved_weights, data_square, 'square')

In [ ]:
u = np.random.uniform(0, 1, n_samples)
v = np.random.uniform(-1, 1, n_samples)
y = 1 - 2 * np.sqrt(u)
x = v * (1 - y) / 2
data_triangle = np.column_stack((x, y))
visualize_dataset(data_triangle)  

In [ ]:
save_epochs = [1, 2, 300, 1000]
model = KohonenNet(n_neurons=(5, 5))
model.train(data_triangle, sigma_sq_start=1.0, lr=0.01, epochs=1000, save_epochs=save_epochs)
visualize(save_epochs, model.saved_weights, data_triangle, 'triangle')

In [ ]:
r1 = 1.0
r2 = 0.5
u1 = np.random.uniform(0, 1, n_samples)
u2 = np.random.uniform(0, 1, n_samples)
theta = 2 * np.pi * u1
r = np.sqrt(u2 * (r1**2 - r2**2) + r2**2)
x = r * np.cos(theta)
y = r * np.sin(theta)
data_ring = np.column_stack((x, y))
visualize_dataset(data_ring)  

In [ ]:
save_epochs = [1, 3, 300, 1000]
model = KohonenNet(n_neurons=(5, 5))
model.train(data_ring, sigma_sq_start=1.0, lr=0.01, epochs=1000, save_epochs=save_epochs)
visualize(save_epochs, model.saved_weights, data_ring, 'ring')

In [ ]:
np.random.seed(78_86_22)

def is_inside_star(x, y, R=1.0):
    r = R * 0.381966 
    radius = np.sqrt(x**2 + y**2)
    angle = np.arctan2(y, x)
    alpha = 2 * np.pi / 5
    angle_mod = angle % alpha
    angle_mod = np.abs(angle_mod - alpha / 2)
    max_r = (r * R * np.cos(alpha / 2)) / (r * np.sin(angle_mod) + R * np.cos(alpha / 2 - angle_mod))
    return radius <= max_r

def sample_single_star(n_samples, R=1.0):
    points = []
    while len(points) < n_samples:
        chunk_size = (n_samples - len(points)) * 2
        x_box = np.random.uniform(-R, R, chunk_size)
        y_box = np.random.uniform(-R, R, chunk_size)
        
        inside_mask = is_inside_star(x_box, y_box, R)
        
        valid_points = np.column_stack((x_box[inside_mask], y_box[inside_mask]))
        points.extend(valid_points)
        
    return np.array(points[:n_samples])
    
R = 1.2
n_total = 1000
spacing = 0.35
counts = np.random.multinomial(n_total, [1/3, 1/3, 1/3])
n_star1, n_star2, n_star3 = counts

star1 = sample_single_star(n_star1, R) if n_star1 > 0 else np.empty((0, 2))
star2 = sample_single_star(n_star2, R) if n_star2 > 0 else np.empty((0, 2))
star3 = sample_single_star(n_star3, R) if n_star3 > 0 else np.empty((0, 2))

if n_star1 > 0: star1[:, 0] -= 1.75*spacing
if n_star1 > 0: star1[:, 1] -= spacing
if n_star2 > 0: star2[:, 1] += spacing
if n_star3 > 0: star3[:, 0] += 1.75*spacing
if n_star3 > 0: star3[:, 1] -= spacing

data_stars = np.vstack((star1, star2, star3))
np.random.shuffle(data_stars)

visualize_dataset(data_stars)

In [ ]:
save_epochs = [1, 2, 300, 1000]
model = KohonenNet(n_neurons=(5, 5))
model.train(data_stars, sigma_sq_start=1.0, lr=0.1, epochs=1000, save_epochs=save_epochs)
visualize(save_epochs, model.saved_weights, data_stars, 'stars')